# B3 – Normalizacija in standardizacija

**Samostojno delo · Telekom**

**Naloge:** prilagodi oba postopka na isti učni množici, preveri učne statistike,
transformiraj test brez ponovnega fit in razloži razlike. Nato po želji zamenjaj
predlagani značilki ter ponovi od njune izbire naprej.


## Priprava

Notebook je samostojen in deluje lokalno ter v Google Colabu. Celice izvajaj od
zgoraj navzdol s **Shift + Enter**. Ob menjavi stolpcev ponovi ustrezne naslednje celice.

Prva celica poišče `utils/orodja.py`; v Colabu po potrebi prenese repozitorij in namesti
knjižnice iz `requirements.txt`. Lokalno jih enkrat namestimo po navodilih v README.
Druga celica pokliče skupni `poskrbi_za_podatke()` in vrne mapo `DATA`.
Funkcija uporabi pripravljene ali že prenesene podatke; manjkajoče prenese s Kaggla.
CSV-je nato beremo iz `DATA`, ne glede na mapo, v kateri je zvezek odprt.
Izvornih vrednosti ne popravljamo in rezultatov vaj ne zapisujemo samodejno v datoteke.

Tukaj **normalizacija pomeni min–max skaliranje stolpcev**.
`sklearn.preprocessing.Normalizer` je drug postopek: normalizira posamezne vrstice oziroma vektorje.


In [ ]:
#@title Nastavitev okolja (kliknite za prikaz kode)
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/valira-ai/ui-za-poslovne-aplikacije.git"

if "google.colab" in sys.modules:
    KOREN = Path("/content/ui-za-poslovne-aplikacije")
    if not (KOREN / "utils/orodja.py").is_file():
        if KOREN.exists():
            raise FileNotFoundError(f"Mapa {KOREN} že obstaja, vendar v njej ni utils/orodja.py.")
        subprocess.run(["git", "clone", "-q", REPO_URL, str(KOREN)], check=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(KOREN / "requirements.txt")],
        check=True,
    )
else:
    KOREN = Path.cwd().resolve()
    while not (KOREN / "utils/orodja.py").is_file() and KOREN != KOREN.parent:
        KOREN = KOREN.parent
    if not (KOREN / "utils/orodja.py").is_file():
        raise FileNotFoundError("Zaženite zvezek v prenesenem repozitoriju ali njegovi podmapi (manjka utils/orodja.py).")

if str(KOREN) not in sys.path:
    sys.path.insert(0, str(KOREN))
print("Gradivo je v:", KOREN)

### Skupni podatkovni vhod
`poskrbi_za_podatke()` iz `utils/orodja.py` pripravi skupno mapo podatkov in jo vrne kot `DATA`.

V tem sklopu beremo `DATA / "telekom_izvirni.csv"`: vseh 100 stolpcev z izvirnimi imeni.
Uvod in vizualizacija uporabljata 22-stolpčni slovenski izbor `telekom.csv`;
obe tabeli izhajata iz istih strank in ju pripravi isti podatkovni vhod.


In [ ]:
from utils.orodja import poskrbi_za_podatke

DATA = poskrbi_za_podatke()

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.preprocessing import MinMaxScaler, RobustScaler, StandardScaler

pd.set_option("display.max_columns", 15)
pd.set_option("display.max_colwidth", 35)
plt.rcParams.update({"figure.figsize": (9, 4), "font.size": 11})

## 1. Izberi meritve in pripravi delitev

| Predlog | Kaj primerjamo? |
|---|---|
| `rev_Mean`, `mou_Mean` | Izvorne denarne enote in minute uporabe; začetni primer |
| `mou_Mean`, `custcare_Mean` | Uporaba in agregirana mera stikov s podporo |
| `rev_Mean`, `months` | Denarna mera in trajanje razmerja v mesecih |
| `hnd_price`, `mou_Mean` | Cena telefona in uporaba |

`Customer_ID` je ključ, `income` in `crclscod` sta kodi, `churn` je cilj. Ne skaliramo
jih avtomatično kot merjene numerične značilke. Denarnih vrednosti ne imenujemo evri,
če njihovih enot nismo preverili. Skaliranje ne razreši pomena negativne vrednosti.

Za ta izolirani prikaz izpustimo le vrstice brez katere od **izbranih** meritev.
To ni splošno čiščenje vseh 100 stolpcev. V celotnem projektu bi po potrebi uporabili
imputacijo, prav tako prilagojeno samo na učnem delu.

Delitev 80/20 je reproducibilna, naključna in didaktična. Ne učimo ali ocenjujemo
modela; v naslednjem sklopu podrobneje izberemo delitev glede na napovedni problem.


In [ ]:
df = pd.read_csv(DATA / "telekom_izvirni.csv")
stolpci = ["rev_Mean", "mou_Mean"]  # Izberi drug predlagani par in ponovi od te celice naprej.
podatki = df.dropna(subset=stolpci).copy()
assert podatki["Customer_ID"].is_unique
print(f"Strank: {len(df):,}; za prikaz obdržimo: {len(podatki):,}; izpuščenih: {len(df)-len(podatki):,}")
display(podatki[stolpci].describe().T.round(3))

ucni_indeksi, testni_indeksi = train_test_split(podatki.index, test_size=0.2, random_state=42)
X_train = podatki.loc[ucni_indeksi, stolpci].copy()
X_test = podatki.loc[testni_indeksi, stolpci].copy()
assert set(podatki.loc[ucni_indeksi, "Customer_ID"]).isdisjoint(
    set(podatki.loc[testni_indeksi, "Customer_ID"])
)
print(f"Učni del: {len(X_train):,}; testni del: {len(X_test):,}")

## 2. Min–max skaliranje

Za nekonstanten stolpec velja `(x − učni minimum) / učni razpon`. Parametra
izračunamo ločeno za vsak stolpec. Najmanjša učna vrednost postane 0, največja 1.
Za konstanten stolpec implementacija posebej obravnava ničelni razpon.

**Preveri:** kaj vsebujejo `data_min_`, `data_max_` in `data_range_`? Kateri podatki
so sodelovali pri njihovem izračunu? ID ni del matrike `X_train`.


In [ ]:
minmax = MinMaxScaler(feature_range=(0, 1), clip=False)
minmax.fit(X_train)  # Samo učni podatki določijo parametre.

train_mm = pd.DataFrame(minmax.transform(X_train), index=X_train.index, columns=stolpci)
# Na testu uporabimo iste parametre. Ne kličemo novega fit.
test_mm = pd.DataFrame(minmax.transform(X_test), index=X_test.index, columns=stolpci)

parametri_mm = pd.DataFrame({
    "učni minimum": minmax.data_min_,
    "učni maksimum": minmax.data_max_,
    "učni razpon": minmax.data_range_,
}, index=stolpci)
display(parametri_mm.round(4))
display(pd.concat({"izvirno": X_train.head(5), "min–max": train_mm.head(5)}, axis=1))

## 3. Standardizacija

`z = (x − učno povprečje) / učni standardni odklon`. Negativna vrednost pomeni
položaj pod učnim povprečjem, ne nujno napake. Rezultat ni omejen na [0, 1].
`fit_transform(X_train)` je okrajšava za učenje parametrov in transformacijo učnih
vrstic. Za testne ali nove podatke uporabimo samo `transform`.


In [ ]:
standard = StandardScaler()
# fit_transform je na učni množici okrajšava za fit in nato transform.
train_std = pd.DataFrame(
    standard.fit_transform(X_train), index=X_train.index, columns=stolpci
)
test_std = pd.DataFrame(standard.transform(X_test), index=X_test.index, columns=stolpci)

parametri_std = pd.DataFrame({
    "učno povprečje": standard.mean_,
    "učni standardni odklon (ddof=0)": np.sqrt(standard.var_),
    "delitelj scale_": standard.scale_,
}, index=stolpci)
display(parametri_std.round(4))
display(pd.concat({"izvirno": X_train.head(5), "standardizirano": train_std.head(5)}, axis=1))

## 4. Preverimo učne statistike

`StandardScaler` uporablja standardni odklon z **`ddof=0`**. Pandasov `.std()`
privzeto uporablja `ddof=1`, zato spodaj izpišemo oba. Pri nekonstantnem učnem
stolpcu pričakujemo povprečje približno 0 in odklon z `ddof=0` približno 1.
Konstanten stolpec je po privzetem centriranju ničeln, njegov odklon ni 1.
Majhni ostanki okoli nič so posledica računanja s plavajočo vejico.


In [ ]:
preverjanje = pd.DataFrame({
    "min–max: minimum": train_mm.min(),
    "min–max: maksimum": train_mm.max(),
    "standard: povprečje": train_std.mean(),
    "standard: std (ddof=0)": train_std.std(ddof=0),
    "standard: std (ddof=1)": train_std.std(ddof=1),
})
display(preverjanje.round(8))

# Kontrola veljavnih učnih lastnosti, tudi če bi izbrali konstanten stolpec.
nekonstantni = X_train.nunique().gt(1).astype(float)
np.testing.assert_allclose(train_mm.min(), 0, atol=1e-10)
np.testing.assert_allclose(train_mm.max(), nekonstantni, atol=1e-10)
np.testing.assert_allclose(train_std.mean(), 0, atol=1e-10)
np.testing.assert_allclose(train_std.std(ddof=0), nekonstantni, atol=1e-10)
print("Kontrole učnih statistik so uspešne.")

## 5. Isti parametri na testnih in novih podatkih

Testne statistike niso nujno enake učnim. Tudi testni min–max ni nujno omejen na
[0, 1], če nova vrednost preseže učni razpon. Pri tej delitvi lahko vse testne
vrednosti ostanejo znotraj intervala; ni razloga, da bi morale nekatere pasti ven.

Spodnji izpis uporablja že izračunane `test_mm` in `test_std`. Na testu ne kličemo
`fit` ali `fit_transform`. Toleranca 1e−12 pri štetju izloči zanemarljive numerične
ostanke na meji intervala.


In [ ]:
primerjava = pd.concat({
    "učni min–max": train_mm.agg(["min", "max", "mean"]),
    "testni min–max": test_mm.agg(["min", "max", "mean"]),
    "učna standardizacija": train_std.agg(["mean", "std"]),
    "testna standardizacija": test_std.agg(["mean", "std"]),
})
# V tej tabeli za std izrecno uporabimo isto konvencijo ddof=0.
primerjava.loc[("učna standardizacija", "std"), :] = train_std.std(ddof=0)
primerjava.loc[("testna standardizacija", "std"), :] = test_std.std(ddof=0)
display(primerjava.round(6))

zunaj = test_mm.lt(-1e-12) | test_mm.gt(1 + 1e-12)
display(pd.DataFrame({
    "testnih vrednosti zunaj [0, 1]": zunaj.sum(),
    "delež (%)": zunaj.mean() * 100,
}).round(4))
print(f"Testnih vrstic z vsaj eno vrednostjo zunaj intervala: {zunaj.any(axis=1).sum():,}")

### Jasno označen primer nove vrednosti izven učnega razpona

Izdelamo **umetno novo vrstico**: prvi stolpec je za 10 % učnega razpona nad učnim
maksimumom. Drugi stolpci so na učnih medianah. Vrstica ni iz zbirke ali testa in
je ne dodajamo učnim podatkom. Ob privzetem `clip=False` prva min–max vrednost
preseže 1. `clip=True` bi jo omejil, ne bi pa odpravil spremembe podatkov.


In [ ]:
# Umeten prikaz nove vrstice; to ni dejanski zapis iz zbirke ali del testa.
novi = X_train.median().to_frame().T
novi.index = ["umetna nova vrstica"]
prvi_stolpec = stolpci[0]
novi.loc[:, prvi_stolpec] = minmax.data_max_[0] + 0.10 * minmax.data_range_[0]

maksimum_pred = minmax.data_max_.copy()
povprecje_pred = standard.mean_.copy()
novi_mm = pd.DataFrame(minmax.transform(novi), index=novi.index, columns=stolpci)
novi_std = pd.DataFrame(standard.transform(novi), index=novi.index, columns=stolpci)
display(pd.concat({"izvirno": novi, "min–max": novi_mm, "standard": novi_std}, axis=1))
np.testing.assert_allclose(minmax.data_max_, maksimum_pred)
np.testing.assert_allclose(standard.mean_, povprecje_pred)
print("Po transformaciji nove vrstice so učni parametri ostali enaki.")

**Vprašanja:** zakaj testno povprečje ni nujno 0? Zakaj vrednost nad 1 ni sama po
sebi napaka pri računanju? Kaj bi izgubili, če bi na vsaki novi skupini podatkov
znova izvedli `fit_transform`?

Naslednja poglavja so dodatni prikazi za povezavo s koncepti in vprašanja skupine.


## 6. Oblika porazdelitve ostane (dodatno)

Min–max in standardizacija sta linearni spremembi lestvice s pozitivnim faktorjem.
Ohranita vrstni red ter obliko porazdelitve nekonstantnega stolpca. Desni rep ne
postane normalna porazdelitev. Logaritemska transformacija je drugačen poseg in
je tukaj ne izvajamo.

Na histogramih pokažemo **iste učne vrstice do 99. percentila izvornega stolpca**,
da vidimo glavnino. Scalerja in izpis asimetrije še vedno uporabljajo vse učne
vrednosti, vključno z ekstremi. Osi imajo različne enote.


In [ ]:
stolpec_prikaza = stolpci[0]  # Za drugi primer izberi stolpci[1].
serije = {
    "Izvirno": X_train[stolpec_prikaza],
    "Min–max": train_mm[stolpec_prikaza],
    "Standardizirano": train_std[stolpec_prikaza],
}
# Asimetrijo in parametre računamo na vseh učnih vrednostih.
display(pd.Series({ime: s.skew() for ime, s in serije.items()}, name="asimetrija (skew)").to_frame())

# Za graf izberemo iste vrstice do 99. percentila izvornega stolpca.
maska_prikaza = X_train[stolpec_prikaza].le(X_train[stolpec_prikaza].quantile(0.99))
fig, osi = plt.subplots(1, 3, figsize=(13, 4))
for os, (ime, s) in zip(osi, serije.items()):
    os.hist(s.loc[maska_prikaza], bins=40, edgecolor="white")
    os.set(title=ime, xlabel=stolpec_prikaza, ylabel="Število zapisov")
fig.suptitle("Iste vrstice in druga merska lestvica; oblika ostane")
fig.tight_layout()
plt.show()

## 7. RobustScaler: mediana in IQR (dodatno)

Privzeta delitelja sta IQR pri `RobustScaler` in standardni odklon pri
`StandardScaler`. Mediana/IQR sta praviloma manj občutljiva na posamezne ekstreme,
vendar robustno skaliranje ekstremov ne odstrani in ne omeji razpona.

Na grafih povečamo območje **od −5 do 5**. Vrednosti zunaj tega prikaza še vedno
obstajajo; maksimum je izpisan v tabeli. IQR po robustni transformaciji je približno
1 le za stolpce z neničelnim IQR in ob prikazanih privzetih nastavitvah.


In [ ]:
robust = RobustScaler(quantile_range=(25, 75), unit_variance=False)
train_robust = pd.DataFrame(robust.fit_transform(X_train), index=X_train.index, columns=stolpci)
test_robust = pd.DataFrame(robust.transform(X_test), index=X_test.index, columns=stolpci)

display(pd.DataFrame({
    "učna mediana": robust.center_,
    "delitelj RobustScaler": robust.scale_,
    "mediana po transformaciji": train_robust.median(),
    "IQR po transformaciji": train_robust.quantile(0.75) - train_robust.quantile(0.25),
    "učni maksimum po transformaciji": train_robust.max(),
}, index=stolpci).round(4))

fig, osi = plt.subplots(1, 2, figsize=(11, 4))
for os, (ime, tabela) in zip(osi, [("StandardScaler", train_std), ("RobustScaler", train_robust)]):
    os.boxplot(tabela[stolpec_prikaza], whis=1.5, flierprops={"markersize": 2, "alpha": 0.25})
    os.set_xticks([])
    os.set(title=ime, ylabel="Skalirana vrednost; različna delitelja", ylim=(-5, 5))
fig.suptitle(f"Povečan prikaz od −5 do 5: {stolpec_prikaza}")
fig.tight_layout()
plt.show()

## 8. Merska lestvica in razdalje (dodatno)

Za začetni par pretvorimo `mou_Mean` iz minut v sekunde (×60). Nato primerjamo
prispevke h kvadrirani evklidski razdalji pred skaliranjem ter po min–max in
standardizaciji. Če si zamenjal par stolpcev, ustrezno prilagodi tudi razlago
in faktor pretvorbe enote v tej dodatni celici.

Scalerja sta prilagojena na vseh učnih vrsticah, ne samo na paru. Enake lestvice ne
pomenijo enake poslovne pomembnosti ali prispevkov 50 : 50 pri vsakem paru.


In [ ]:
# Dve različni učni vrstici sta le primer; scalerja sta prilagojena na vseh učnih vrsticah.
par = X_train.drop_duplicates().head(2)
par_mm = pd.DataFrame(minmax.transform(par), index=par.index, columns=stolpci)
par_std = pd.DataFrame(standard.transform(par), index=par.index, columns=stolpci)
spremenjene_enote = par.copy()
stolpec_enote = stolpci[1]
faktor_enote = 60
spremenjene_enote[stolpec_enote] = spremenjene_enote[stolpec_enote] * faktor_enote

kvadrati_razlik = pd.DataFrame({
    "izvorno": (par.iloc[1] - par.iloc[0]) ** 2,
    "druga enota": (spremenjene_enote.iloc[1] - spremenjene_enote.iloc[0]) ** 2,
    "min–max": (par_mm.iloc[1] - par_mm.iloc[0]) ** 2,
    "standardizacija": (par_std.iloc[1] - par_std.iloc[0]) ** 2,
})
vsote = kvadrati_razlik.sum(axis=0)
prispevki = kvadrati_razlik.div(vsote.replace(0, np.nan), axis=1) * 100
print(f"Za prikaz druge enote pomnožimo {stolpec_enote} s {faktor_enote}.")
display(par)
display(prispevki.T.round(2))
display(pd.DataFrame({"kvadrirana razdalja": vsote, "evklidska razdalja": np.sqrt(vsote)}))

fig, os = plt.subplots()
prispevki.T.plot.bar(stacked=True, ax=os)
os.set(ylabel="Prispevek h kvadrirani razdalji (%)", xlabel="Predstavitev podatkov",
       title="Drugačne enote in skaliranje spremenijo prispevke", ylim=(0, 100))
os.legend(title="Značilka", bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
plt.show()